# Final Project Draft: Tottenham Hotspur Recruitment Fit Model

This draft prototype builds a machine learning workflow for identifying players who fit Tottenham Hotspur's recruitment profile. The original proposal aimed to combine FBref and Transfermarkt data across multiple leagues; for the draft, I narrowed the scope to a clean multi-season Premier League player dataset so I could validate the full pipeline before expanding to the final version.

The notebook uses four seasons of Fantasy Premier League player data as a reproducible player-level source, then engineers a continuous `tottenham_fit_score` that blends style similarity, affordability, and availability. I compare two techniques from my proposal: K-Means clustering for player archetypes and Random Forest regression for fit-score prediction.

## 1. Data Loading

**Draft dataset choice:** Fantasy Premier League historical player data from the public `vaastav/Fantasy-Premier-League` repository.

**Why this dataset works for the draft:**
- It contains player-level observations across multiple Premier League seasons.
- It includes enough numerical features to model style, output, and cost proxy.
- It allows me to test the Tottenham fit concept end-to-end before expanding to FBref plus Transfermarkt in the final submission.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42

In [ ]:
season_paths = ['2021-22', '2022-23', '2023-24', '2024-25']
position_map = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
frames = []

for season in season_paths:
    players = pd.read_csv(Path('data/fpl_raw') / season / 'players_raw.csv')
    teams = pd.read_csv(Path('data/fpl_raw') / season / 'teams.csv')[
        ['id', 'name', 'short_name', 'strength', 'strength_attack_home', 'strength_defence_home']
    ].rename(columns={'id': 'team', 'name': 'team_name'})

    season_df = players.merge(teams, on='team', how='left')
    season_df['season'] = season
    season_df['position'] = season_df['element_type'].map(position_map)
    frames.append(season_df)

raw_df = pd.concat(frames, ignore_index=True)

for col in ['creativity', 'influence', 'threat', 'ict_index', 'points_per_game', 'selected_by_percent', 'form']:
    raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')

raw_df['player_name'] = raw_df['first_name'] + ' ' + raw_df['second_name']
raw_df['is_spurs'] = (raw_df['short_name'] == 'TOT').astype(int)

# Keep outfield players with enough minutes to represent a meaningful season sample.
df = raw_df[raw_df['position'].isin(['DEF', 'MID', 'FWD'])].copy()
df = df[df['minutes'] >= 270].copy()

print(f'Observations after filtering: {df.shape[0]}')
print(f'Features before engineering: {df.shape[1]}')
display(df[['player_name', 'season', 'team_name', 'position', 'minutes', 'goals_scored', 'assists', 'now_cost']].head())

## 2. Exploratory Analysis

Before modeling, I want to understand how large the dataset is, which variables are available, whether any values are missing, and how Tottenham players compare to the rest of the league.

In [ ]:
print('Dataset shape:', df.shape)
print('Seasons included:', sorted(df['season'].unique().tolist()))
print('Positions included:')
print(df['position'].value_counts())
print('
Tottenham observations:', int(df['is_spurs'].sum()))
print('Non-Tottenham observations:', int((1 - df['is_spurs']).sum()))

summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_values': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values(['missing_values', 'dtype'], ascending=[False, True])

display(summary.head(20))

In [ ]:
eda_numeric = [
    'minutes', 'goals_scored', 'assists', 'creativity', 'influence', 'threat',
    'ict_index', 'now_cost', 'points_per_game', 'form'
]

display(df[eda_numeric].describe().T[['mean', 'std', 'min', 'max']])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x='position', hue='is_spurs', palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Position Mix: Spurs vs Non-Spurs')
axes[0, 0].set_xlabel('Position')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend(title='Is Spurs', labels=['No', 'Yes'])

sns.boxplot(data=df, x='is_spurs', y='creativity', palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title('Creativity Distribution by Spurs Label')
axes[0, 1].set_xlabel('Is Spurs')
axes[0, 1].set_ylabel('Creativity')
axes[0, 1].set_xticklabels(['No', 'Yes'])

sns.scatterplot(
    data=df,
    x='influence',
    y='threat',
    hue='is_spurs',
    palette='Set1',
    alpha=0.7,
    ax=axes[1, 0]
)
axes[1, 0].set_title('Influence vs Threat')
axes[1, 0].set_xlabel('Influence')
axes[1, 0].set_ylabel('Threat')

team_cost = df.groupby('is_spurs')['now_cost'].mean().rename({0: 'Other clubs', 1: 'Spurs'})
team_cost.plot(kind='bar', color=['#4C72B0', '#55A868'], ax=axes[1, 1])
axes[1, 1].set_title('Average Price Proxy by Group')
axes[1, 1].set_xlabel('Group')
axes[1, 1].set_ylabel('Average now_cost')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

**Key data characteristics:**

The draft dataset contains more than 1,600 outfield player-season observations across four Premier League seasons, which is enough to test the recruitment concept at prototype scale. Missingness is low in the selected modeling columns, and Tottenham players represent a small minority of the dataset, so this project is better framed as a fit-score problem than a simple club-membership classification problem. The feature distributions also show strong differences by position, which means position-aware preprocessing is important when defining player similarity.

## 3. Feature Engineering

To make the raw football statistics more useful for recruitment modeling, I engineered features that capture output efficiency, style balance, and price-adjusted availability.

In [ ]:
# Engineered feature 1: attacking output efficiency.
df['goal_involvement_per90'] = ((df['goals_scored'] + df['assists']) / df['minutes'].clip(lower=1)) * 90

# Engineered feature 2: how creative a player is relative to direct goal threat.
df['creativity_threat_balance'] = df['creativity'] / (df['threat'] + 1)

# Engineered feature 3: contribution quality per 90 minutes.
df['bps_per90'] = (df['bps'] / df['minutes'].clip(lower=1)) * 90

# Engineered feature 4: simple affordability and reliability proxy.
df['availability_value_index'] = df['minutes'] / df['now_cost'].clip(lower=1)

engineered_cols = [
    'goal_involvement_per90',
    'creativity_threat_balance',
    'bps_per90',
    'availability_value_index'
]

display(df[['player_name', 'season', 'position'] + engineered_cols].head())

**Feature justifications:**

1. `goal_involvement_per90`: This measures how often a player contributes directly to goals relative to playing time. It should help separate players with genuine attacking efficiency from players who only accumulate raw totals because they played many minutes.

2. `creativity_threat_balance`: This ratio helps distinguish creators from direct finishers. That matters in recruitment because Tottenham may want different player archetypes across midfield and forward roles, not just players with high totals.

3. `bps_per90`: Bonus point system score captures all-around contribution in the FPL source data. Converting it to a per-90 rate helps compare players more fairly across different minute totals.

4. `availability_value_index`: This combines durability and affordability by relating minutes played to the player's current price proxy. It supports the proposal's emphasis on realistic, cost-aware recruitment rather than selecting expensive names only.

## 4. Create the Tottenham Fit Target

For the draft, I engineered a continuous `tottenham_fit_score` instead of using an external label. The score combines:
- style similarity to historical Tottenham players in the same position,
- affordability using the FPL price field as a draft proxy for market value,
- availability based on minutes played.

This is a prototype version of the proposal's final fit score. In the full project, I would replace the price proxy with Transfermarkt market values and expand the player pool beyond the Premier League.

In [ ]:
style_cols = ['creativity', 'influence', 'threat', 'ict_index', 'goal_involvement_per90', 'bps_per90', 'clean_sheets']

# Position-wise z-scores reduce the bias created by comparing defenders directly with forwards.
for col in style_cols:
    group_stats = df.groupby('position')[col]
    df[f'{col}_z'] = (df[col] - group_stats.transform('mean')) / group_stats.transform('std').replace(0, np.nan)

z_cols = [f'{col}_z' for col in style_cols]
spurs_centroids = df[df['is_spurs'] == 1].groupby('position')[z_cols].mean()

def cosine_similarity(vec_a, vec_b):
    vec_a = np.asarray(vec_a, dtype=float)
    vec_b = np.asarray(vec_b, dtype=float)
    if np.isnan(vec_a).any() or np.isnan(vec_b).any():
        return np.nan
    denom = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if denom == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / denom)

style_similarity_raw = []
for _, row in df.iterrows():
    centroid = spurs_centroids.loc[row['position']]
    style_similarity_raw.append(cosine_similarity(row[z_cols], centroid))

df['style_similarity_raw'] = style_similarity_raw

def minmax_0_100(series):
    series = series.astype(float)
    min_val, max_val = series.min(), series.max()
    if max_val - min_val == 0:
        return pd.Series(50.0, index=series.index)
    return (series - min_val) / (max_val - min_val) * 100

df['style_similarity_score'] = minmax_0_100(df['style_similarity_raw'])
df['affordability_score'] = minmax_0_100(-df['now_cost'])
df['availability_score'] = minmax_0_100(df['minutes'])

df['tottenham_fit_score'] = (
    0.65 * df['style_similarity_score'] +
    0.20 * df['affordability_score'] +
    0.15 * df['availability_score']
)

print('Fit score range:')
print(df['tottenham_fit_score'].agg(['min', 'max', 'mean', 'std']).round(2))
display(
    df[['player_name', 'season', 'team_name', 'position', 'style_similarity_score', 'affordability_score', 'availability_score', 'tottenham_fit_score']]
    .sort_values('tottenham_fit_score', ascending=False)
    .head(10)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df['tottenham_fit_score'], bins=25, kde=True, ax=axes[0], color='#4C72B0')
axes[0].set_title('Distribution of Tottenham Fit Score')
axes[0].set_xlabel('Fit Score')

sns.boxplot(data=df, x='position', y='tottenham_fit_score', palette='Set3', ax=axes[1])
axes[1].set_title('Fit Score by Position')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Fit Score')

sns.scatterplot(
    data=df,
    x='goal_involvement_per90',
    y='tottenham_fit_score',
    hue='is_spurs',
    palette='Set1',
    alpha=0.7,
    ax=axes[2]
)
axes[2].set_title('Goal Involvement vs Fit Score')
axes[2].set_xlabel('Goal Involvement per 90')
axes[2].set_ylabel('Fit Score')

plt.tight_layout()
plt.show()

## 5. Prepare Data for Modeling

I used an 80/20 train-test split and stratified by position so the training and test sets keep similar role distributions. Numeric features are median-imputed and standardized, while categorical variables (`position` and `season`) are one-hot encoded. Using the same preprocessing pipeline for both models keeps the comparison fair.

In [ ]:
model_features = [
    'goals_scored', 'assists', 'clean_sheets', 'creativity', 'influence', 'threat', 'ict_index',
    'bonus', 'bps', 'minutes', 'now_cost', 'points_per_game', 'selected_by_percent', 'form',
    'goal_involvement_per90', 'creativity_threat_balance', 'bps_per90', 'availability_value_index',
    'strength', 'strength_attack_home', 'strength_defence_home', 'position', 'season'
]

X = df[model_features].copy()
y = df['tottenham_fit_score'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df['position']
)

numeric_features = [col for col in model_features if col not in ['position', 'season']]
categorical_features = ['position', 'season']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]),
            numeric_features
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Training positions:')
print(X_train['position'].value_counts())

## 6. Model 1: K-Means Clustering for Player Archetypes

K-Means is used here as the player-profiling method from my proposal. To turn clustering into a draft evaluation baseline, I assign each player a cluster and then use the training-set average fit score of that cluster as the predicted fit score for players in the test set.

In [ ]:
silhouette_results = []
for k in range(3, 9):
    candidate_model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    cluster_labels = candidate_model.fit_predict(X_train_processed)
    silhouette_results.append({'k': k, 'silhouette_score': silhouette_score(X_train_processed, cluster_labels)})

silhouette_df = pd.DataFrame(silhouette_results)
best_k = int(silhouette_df.sort_values('silhouette_score', ascending=False).iloc[0]['k'])

plt.figure(figsize=(7, 4))
sns.lineplot(data=silhouette_df, x='k', y='silhouette_score', marker='o')
plt.title('Choosing K with Silhouette Score')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette score')
plt.show()

print('Best k selected for K-Means:', best_k)
display(silhouette_df)

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20)
train_clusters = kmeans.fit_predict(X_train_processed)
test_clusters = kmeans.predict(X_test_processed)

cluster_fit_lookup = (
    pd.DataFrame({'cluster': train_clusters, 'fit_score': y_train.reset_index(drop=True)})
    .groupby('cluster')['fit_score']
    .mean()
)

kmeans_predictions = pd.Series(test_clusters).map(cluster_fit_lookup).to_numpy()

kmeans_rmse = mean_squared_error(y_test, kmeans_predictions) ** 0.5
kmeans_mae = mean_absolute_error(y_test, kmeans_predictions)
kmeans_r2 = r2_score(y_test, kmeans_predictions)

print('K-Means baseline metrics')
print('RMSE:', round(kmeans_rmse, 4))
print('MAE:', round(kmeans_mae, 4))
print('R^2:', round(kmeans_r2, 4))

In [ ]:
train_cluster_df = X_train.copy()
train_cluster_df['cluster'] = train_clusters
train_cluster_df['fit_score'] = y_train.values
train_cluster_df['is_spurs'] = df.loc[X_train.index, 'is_spurs'].values

cluster_summary = train_cluster_df.groupby('cluster').agg(
    players=('cluster', 'size'),
    avg_fit_score=('fit_score', 'mean'),
    spurs_share=('is_spurs', 'mean')
).sort_values('avg_fit_score', ascending=False)
cluster_summary['spurs_share'] = (cluster_summary['spurs_share'] * 100).round(2)
display(cluster_summary)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
train_pca = pca.fit_transform(X_train_processed)
plot_df = pd.DataFrame({
    'pc1': train_pca[:, 0],
    'pc2': train_pca[:, 1],
    'cluster': train_clusters,
    'is_spurs': train_cluster_df['is_spurs'].values
})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df, x='pc1', y='pc2', hue='cluster', style='is_spurs', alpha=0.75, palette='tab10')
plt.title('K-Means Player Archetypes (PCA Projection)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

## 7. Model 2: Random Forest Regression

Random Forest regression is the supervised technique from my proposal. It is a strong fit here because the recruitment score depends on nonlinear relationships between creativity, threat, minutes, cost, and broader contribution metrics.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=4,
    random_state=RANDOM_STATE
)

start_time = time.time()
rf.fit(X_train_processed, y_train)
rf_train_time = time.time() - start_time

rf_predictions = rf.predict(X_test_processed)

rf_rmse = mean_squared_error(y_test, rf_predictions) ** 0.5
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_r2 = r2_score(y_test, rf_predictions)

print('Random Forest metrics')
print('RMSE:', round(rf_rmse, 4))
print('MAE:', round(rf_mae, 4))
print('R^2:', round(rf_r2, 4))
print('Training time (s):', round(rf_train_time, 4))

In [ ]:
comparison_plot = pd.DataFrame({
    'actual_fit_score': y_test.values,
    'predicted_fit_score': rf_predictions
})

plt.figure(figsize=(6, 6))
sns.scatterplot(data=comparison_plot, x='actual_fit_score', y='predicted_fit_score', alpha=0.7)
plt.plot([comparison_plot.min().min(), comparison_plot.max().max()],
         [comparison_plot.min().min(), comparison_plot.max().max()],
         color='red', linestyle='--')
plt.title('Random Forest: Predicted vs Actual Fit Score')
plt.xlabel('Actual fit score')
plt.ylabel('Predicted fit score')
plt.show()

## 8. Compare Model Performance

Both models use the same target, train-test split, and preprocessing. That makes it possible to compare them directly using regression metrics.

In [ ]:
results = pd.DataFrame([
    {
        'Model': 'K-Means cluster baseline',
        'Key hyperparameters': f'k={best_k}, n_init=20',
        'RMSE': kmeans_rmse,
        'MAE': kmeans_mae,
        'R2': kmeans_r2,
        'Training time (s)': np.nan,
        'Interpretability': 'High for archetypes',
        'Overfitting signs': 'Low, but limited predictive precision'
    },
    {
        'Model': 'Random Forest Regressor',
        'Key hyperparameters': 'n_estimators=300, max_depth=10, min_samples_leaf=4',
        'RMSE': rf_rmse,
        'MAE': rf_mae,
        'R2': rf_r2,
        'Training time (s)': rf_train_time,
        'Interpretability': 'Moderate via feature importance',
        'Overfitting signs': 'Monitor with learning curve'
    }
])

metric_cols = ['RMSE', 'MAE', 'R2', 'Training time (s)']
results_display = results.copy()
results_display[metric_cols] = results_display[metric_cols].round(4)
display(results_display)

**Model comparison analysis:**

Random Forest performed much better than the K-Means baseline on the held-out test set, with lower RMSE and MAE and a much higher R-squared value. That makes sense because K-Means is useful for discovering player archetypes, but it is not optimized for precise fit-score prediction. The trade-off is that K-Means is easier to explain tactically, while Random Forest is stronger as a prediction engine. For the final submission, I am leaning toward Random Forest as the main model while keeping K-Means as the supporting tactical-profile method because together they match the business goal better than either method alone.

## 9. Training and Validation Visualization

To check whether the Random Forest model is learning useful patterns without extreme overfitting, I plotted a learning curve using cross-validated MAE.

In [ ]:
train_sizes, train_scores, validation_scores = learning_curve(
    estimator=RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=4,
        random_state=RANDOM_STATE
    ),
    X=X_train_processed,
    y=y_train,
    cv=5,
    scoring='neg_mean_absolute_error',
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=1
)

train_mae = -train_scores.mean(axis=1)
validation_mae = -validation_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mae, marker='o', label='Training MAE')
plt.plot(train_sizes, validation_mae, marker='o', label='Validation MAE')
plt.title('Random Forest Learning Curve')
plt.xlabel('Training examples')
plt.ylabel('Mean Absolute Error')
plt.legend()
plt.show()

In [ ]:
feature_names = preprocessor.get_feature_names_out()
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

display(feature_importance.head(15))

plt.figure(figsize=(9, 6))
sns.barplot(data=feature_importance.head(12), x='importance', y='feature', palette='Blues_r')
plt.title('Top Random Forest Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## 10. Example Recruitment Output

A scouting model is only useful if it produces actionable recommendations. The table below shows the highest-scoring non-Spurs players in the draft dataset based on the Random Forest model's predicted fit score.

In [ ]:
all_processed = preprocessor.transform(X)
df['predicted_fit_score_rf'] = rf.predict(all_processed)

recommendations = (
    df[df['is_spurs'] == 0][
        ['player_name', 'season', 'team_name', 'position', 'now_cost', 'goal_involvement_per90', 'predicted_fit_score_rf']
    ]
    .sort_values('predicted_fit_score_rf', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

recommendations['predicted_fit_score_rf'] = recommendations['predicted_fit_score_rf'].round(2)
recommendations['goal_involvement_per90'] = recommendations['goal_involvement_per90'].round(2)
display(recommendations)

## 11. Reflection and Next Steps

**Feature engineering plans:** For the final version, I want to add at least two more features that are closer to the original proposal. The biggest improvements would come from replacing the FPL price proxy with real Transfermarkt market values and adding passing-progression or defensive-action features from FBref so the model can distinguish tactical fit more accurately. I also want to build position-specific fit scores instead of using one general formula across all outfield players.

**Model optimization plans:** Random Forest is the stronger draft model, so the next step is systematic tuning of `max_depth`, `min_samples_leaf`, and `n_estimators` with cross-validation. I also want to test gradient boosting or XGBoost in the final version, then compare those results against the tuned Random Forest. On the clustering side, I want to experiment with position-specific K-Means or Gaussian mixture models to see whether the player archetypes become more interpretable.

**Questions for instructor feedback:** My main question is whether the engineered fit score is reasonable for the draft phase, since it is not a directly observed label. I would also like feedback on whether narrowing the draft to Premier League data is acceptable as a prototype for a final project that will later expand to FBref plus Transfermarkt and include broader league coverage. Finally, I would appreciate guidance on how much emphasis to place on tactical interpretability versus raw predictive performance in the final submission.